In [ ]:
import sys,os,time 
import h5py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import KIPD_Analysis as kipd

# LED Data vs Temperature: Feb 15-16, 2023

In [ ]:
datapath = '/data/USRP_Laser_TempScan_Data' # '/Users/dtemples/Downloads/laser-temp-scans' # 

## Define the data we care about
V_led    = [3.00 , 4.00 , 5.00]
RF_power = -30.0
day_str  = "20230216" # "20230215" # 

In [ ]:
do_noise_cleaning = False
do_pulse_cleaning = False
get_average_pulses = False

use_mapped_temperature = True

### Define some constants

In [ ]:
## PSD hi and lo frequency limits
PSD_lo_f_Hz = 1e1
PSD_hi_f_Hz = 1e6

## Fraction of timestream to trim from beginning (Transient period)
blank_fraction   = 0.1

## Fraction of pulse window to keep for template/noise size
fraction_to_keep = 0.5

## Quadrature for analysis
PHASE = True

## Removal decimation for cleaning (more == better high-F cleaning)
removal_dec    = 1 ## Is this needed ?

In [ ]:
## MB Results come from fitting a temperature scan at a specified RF power
MB_fit_vals = np.array([4.24198 ,  ## Fr(0) [GHz]
                        0.184   ,  ## Delta [meV]
                        0.03801 ,  ## alpha
                        3.7e5   ]) ## Qi(0)
    
## How much attenuation is in the lines before the chip
line_atten_dB = 56.5

### Pull the file index information

In [ ]:
idx_name = "_run_temperatures.csv"

data_idx = pd.read_csv( os.path.join(datapath,day_str,day_str+idx_name), header=None, names=["series","sp K","Ti mK","Tf mK"], skiprows=12 )
# data_idx #= data_idx.loc[12:]

In [ ]:
# data_idx = data_idx.loc[data_idx["sp K"] < 0.201]
# data_idx = data_idx.loc[data_idx["sp K"] > 0.016]
N_runs = len(data_idx["sp K"])
data_idx

### Load the temperature map

In [ ]:
Temp_map = pd.read_hdf("/data/Misc/20230413_TempMap.h5", key="map")
# Temp_map = pd.read_hdf("/Users/dtemples/Downloads/20230413_TempMap.h5", key="map")

In [ ]:
Temp_map ## Each column is a temperature measured by MC, first entry is mean Lakestore temperature, second is Lakeshore temperature sdev

In [ ]:
Temp_map.keys()

In [ ]:
def get_Temp_correction(T_mK):
    
    if T_mK in Temp_map.keys():
        return Temp_map[T_mK][0] , Temp_map[T_mK][1]
    else:
        T_hi = T_mK+5.0
        T_lo = T_mK-5.0
        TLS_hi = Temp_map[T_hi]
        TLS_lo = Temp_map[T_lo]
        
        T_mean = np.mean([TLS_hi[0], TLS_lo[0]])
        T_sdev = np.sqrt( TLS_hi[1]*TLS_hi[1] + TLS_lo[1]*TLS_lo[1] )
        return T_mean , T_sdev
    

## Clean the Noise Data

In [ ]:
if do_noise_cleaning:
    for i in np.arange(N_runs):
        ## Pull the series information
        series = data_idx["series"].loc[i]
        T_sp_K = data_idx["sp K"].loc[i]

        ## Get the file lists
        sum_file, dly_file, vna_file, nse_files, led_files = kipd.get_files(series, 
                                                            base_path=datapath,
                                                            sep_noise_pulse=True,
                                                            verbose=False)

        ## Parse the metadata
        voltages, p_params, charFs, charZs = kipd.parse_calibration_metadata(sum_file, 
                                                            blank_fraction=blank_fraction, 
                                                            verbose=False)

        ## Clean the noise files
        for jj, nse_file in enumerate(nse_files):
            ## Do the cleaning of the noise file
            _, _, _, _ = kipd.cleanPSDs(nse_file, vna_file, 
                PSD_lo_f=PSD_lo_f_Hz, 
                PSD_hi_f=PSD_hi_f_Hz, 
                rem_dec=removal_dec,
                f_transient=p_params['blank_fraction'], 
                charFs=charFs[jj].real if len(np.shape(charFs))>1 else charFs.real, 
                charZs=charZs[jj] if len(np.shape(charFs))>1 else charZs,
                verbose=False,show_plots=False)

        del _

## Clean the Pulse Data

In [ ]:
if do_pulse_cleaning:
    for i in np.arange(N_runs):
        ## Pull the series information
        series = data_idx["series"].loc[i]
        T_sp_K = data_idx["sp K"].loc[i]

        ## Get the file lists
        sum_file, dly_file, vna_file, nse_files, led_files = kipd.get_files(series, 
                                                            base_path=datapath,
                                                            sep_noise_pulse=True,
                                                            verbose=False)

        ## Parse the metadata
        voltages, p_params, charFs, charZs = kipd.parse_calibration_metadata(sum_file, 
                                                            blank_fraction=blank_fraction, 
                                                            verbose=False)

        ## Look at the pulse windows
        pulse_rqs = kipd.extract_rqs_from_files(led_files, p_params, show_plots=False)

        ## Define and apply the cuts, just use default for now
        cut_df       = kipd.define_default_cuts(led_files, pulse_rqs, force_save=True)
        bad_pls_idxs = kipd.apply_cuts_to_all_files(led_files, cut_df, pulse_rqs)

        ## Clean the pulse windows
        kipd.clean_pulse_files(led_files, nse_files[0], p_params, pulse_rqs, bad_pls_idxs, show_plots=False)

## Determine the Average Pulse

In [ ]:
if get_average_pulses:
    for i in np.arange(N_runs):
        ## Pull the series information
        series = data_idx["series"].loc[i]
        T_sp_K = data_idx["sp K"].loc[i]

        ## Get the file lists
        sum_file, dly_file, vna_file, nse_files, led_files = kipd.get_files(series, 
                                                            base_path=datapath,
                                                            sep_noise_pulse=True,
                                                            verbose=False)

        ## Parse the metadata
        voltages, p_params, charFs, charZs = kipd.parse_calibration_metadata(sum_file, 
                                                            blank_fraction=blank_fraction, 
                                                            verbose=False)

        ## Look at the pulse windows
        pulse_rqs = kipd.extract_rqs_from_files(led_files, p_params, show_plots=False)

        ## Define and apply the cuts, just use default for now
        cut_df       = kipd.define_default_cuts(led_files, pulse_rqs, force_save=True)
        bad_pls_idxs = kipd.apply_cuts_to_all_files(led_files, cut_df, pulse_rqs)

        ## Get the average pulse shape
        kipd.get_all_average_pulses(led_files, vna_file, p_params, bad_pls_idxs, extra_decimation=1, show_plots=False, verbose=False)

In [ ]:
# %matplotlib notebook

In [ ]:
AAA = 3.00000000
np.array(led_files)[[ (f"{AAA:.3f}V" in file) for file in led_files ]][0]

In [ ]:
## Create a plot for the average pulses in idealized S21 space
fig = plt.figure(figsize=(8,6), dpi=300)
ax0 = fig.gca()
ax0.set_xlabel(r"$\Re(S_{21})$")
ax0.set_ylabel(r"$\Im(S_{21})$")
ax0.set_title("Average Pulses in Ideal S21")

## Create a pandas df that we will write the average pulses to
f_avg_name = data_idx['series'].iloc[0].split('_')[0]+"_avg_pulses_Vled_"+str(V_led)+".h5"
f_path     = os.path.join(datapath, data_idx['series'].iloc[0].split('_')[0], f_avg_name)

avg_df = pd.DataFrame()
cplx_df = pd.DataFrame()
freq_df = pd.DataFrame()
diss_df = pd.DataFrame()
kappa1_df = pd.DataFrame()
kappa2_df = pd.DataFrame()

temps_mK = []

times_saved = False

temps_to_skip = [350.0, 100.0, 15.0] ## for plotting (MC Temperatures)
temp_to_plot_max = 249.0

this_V_LED = 3.0

for i in np.arange(N_runs):
    ## Pull the series information
    series = data_idx["series"].loc[i]
    temps_mK.append(np.round(1e3 * data_idx["sp K"].loc[i],0))
    
    ## Define the temperature to use
    T_value_K = temps_mK[-1]*1e-3 if not use_mapped_temperature else get_Temp_correction(temps_mK[-1])[0]*1e-3
    
    ## Get the file lists
    sum_file, dly_file, vna_file, nse_files, led_files = kipd.get_files(series, 
                                                        base_path=datapath,
                                                        sep_noise_pulse=True,
                                                        verbose=False)
    
    ## Parse the metadata
    voltages, p_params, charFs, charZs = kipd.parse_calibration_metadata(sum_file, 
                                                        blank_fraction=blank_fraction, 
                                                        verbose=False)
    
    ## Pull the VNA data
    VNA_f, VNA_z = kipd.read_vna(vna_file)
    
    ## Find the right LED file
    # led_file = led_files[np.argmin(np.abs(np.array(voltages)-V_led))]
    led_file = np.array(led_files)[[ (f"{this_V_LED:.3f}V" in file) for file in led_files ]][0]
    print(led_file.split('/')[-1], temps_mK[-1], p_params["rf_power"])
    
    if not (p_params["rf_power"] == RF_power):
        i += 1
        continue
    
    ## Open the cleaned data and pull the data sampling rate
    clean_pulse_file   = led_file[:-3] + '_cleaned.h5'
    pulse_avg          = kipd.get_avgpulse_shape_from_file(clean_pulse_file)
    _, clean_data_info = kipd.get_cleaned_timestream(clean_pulse_file)
    
    sampling_rate = clean_data_info['sampling_rate']

    ## Define the time window for the pulse-full region, in microseconds
    time_window_range = fraction_to_keep * p_params['time_btw_pulse'] *1e6
    time_window = np.arange(0,time_window_range,1/sampling_rate*1e6)#[:-1]
    
    ## Convert the average pulses into the resonator basis
    frequency, dissipation, ideal, resonator = kipd.resonator_basis(pulse_avg,
        VNA_f=VNA_f,
        VNA_z=VNA_z,
        readout_f=charFs[0][0].real if len(np.shape(charFs))>1 else charFs[0].real,
        char_f=charFs[0].real if len(np.shape(charFs))>1 else charFs.real, 
        char_z=charZs[0] if len(np.shape(charFs))>1 else charZs)
    
    ## Get the color for this temperature
#     color = kipd.get_color( T_value_K*1e3 , np.max(
#         Temp_map.keys()
#         if not use_mapped_temperature else
#         data_idx["sp K"]*1e3
#     ), scale_min=0.0, offset=50.0)
    
    color = kipd.c_wheel_0[i % len(kipd.c_wheel_0)]
    
    if (temps_mK[-1] not in temps_to_skip) and (temps_mK[-1] < temp_to_plot_max):
        ## Define the label for this temperature
        label = ( r'$T_\mathrm{MC}=$'+str(int(T_value_K*1e3))+" mK" 
                 if not use_mapped_temperature else 
                  r'$T_\mathrm{RF}=$'+str(int(T_value_K*1e3))+" mK" )

        ## Add the average pulse to complex S21 plot
        ax0.plot( ideal['timestream'].real , ideal['timestream'].imag , c=color , marker='.', ls='None', label=label)
        ax0.plot( ideal['z'].real , ideal['z'].imag , c=color , marker='None', ls='-')
    
    ## Convert the average pulses into the quasiparticle basis
    dnqp_k1, dnqp_k2 = kipd.quasiparticle_basis(frequency,dissipation,
        data_T= T_value_K,
        MB_results=MB_fit_vals,
        readout_f=charFs[0][0].real if len(np.shape(charFs))>1 else charFs[0].real)
    
    ## Make names
    t_vals = np.arange(len(pulse_avg))/sampling_rate*1e3
    p_vals = np.angle(pulse_avg)-np.mean(np.angle(pulse_avg)[0:100])
    
    ## Save the average pulse
    if not times_saved:
        avg_df["time(ms)"] = t_vals
        cplx_df["time(ms)"] = t_vals
        freq_df["time(ms)"] = t_vals
        diss_df["time(ms)"] = t_vals
        kappa1_df["time(ms)"] = t_vals
        kappa2_df["time(ms)"] = t_vals
        times_saved = True
        
    avg_df[ str(int(temps_mK[i]))] = p_vals
    cplx_df[str(int(temps_mK[i]))] = pulse_avg
    freq_df[str(int(temps_mK[i]))] = frequency   -np.mean(frequency[0:100])
    diss_df[str(int(temps_mK[i]))] = dissipation -np.mean(dissipation[0:100])
    kappa1_df[str(int(temps_mK[i]))] = dnqp_k1   -np.mean(dnqp_k1[0:100])
    kappa2_df[str(int(temps_mK[i]))] = dnqp_k2   -np.mean(dnqp_k2[0:100])
    
ax0.legend(loc='center left', bbox_to_anchor=(1, 0.5))

xlim = ax0.get_xlim()
ylim = ax0.get_ylim()

ax0.set_xlim([xlim[0],0.70])
ax0.set_ylim([-0.05, 0.05])

# avg_df.to_hdf(f_path,"data")
# cplx_df.to_hdf(f_path,"s21_cplx")
# freq_df.to_hdf(f_path,"freq")
# diss_df.to_hdf(f_path,"diss")
# kappa1_df.to_hdf(f_path,"k1")
# kappa2_df.to_hdf(f_path,"k2")

In [ ]:
## Instantiate an H5 file
of_name = 'AvgPulsesVsTemp_'+data_idx['series'].iloc[0].split('_')[0]+'.h5'
f_path  = os.path.join(datapath, data_idx['series'].iloc[0].split('_')[0], of_name)
print(f_path)
with h5py.File(f_path, 'w') as outfile:
    
    ## Save the MB data into the H5 attributes
    outfile.attrs["Fr0(GHz)"] = MB_fit_vals[0]
    outfile.attrs["D0(meV)"] = MB_fit_vals[1]
    outfile.attrs["alpha"] = MB_fit_vals[2]
    outfile.attrs["Qi0"] = MB_fit_vals[3]
    outfile.attrs["tot_atten(dB)"] = line_atten_dB
    
    ## Create a group for each LED voltage
    Vled_grps = [outfile.create_group(str(V)+"V") for V in V_led]
    
    ## Save all the data from each LED voltage to the appropriate group
    for i in np.arange(len(V_led)):
    
        ## Pull the voltage and the group
        this_V   = V_led[i]
        this_grp = Vled_grps[i]
    
        ## Save the voltage info to the group as an attribute
        this_grp.attrs["voltage"] = this_V
    
        ## Create some data containers and flags
        temps_mK = []
    
        for j in np.arange(N_runs):
            ## Pull the series information
            series = data_idx["series"].loc[j]
            temps_mK.append(np.round(1e3 * data_idx["sp K"].loc[j],0))
            
            ## Define the temperature to use
            T_value_K = temps_mK[-1]*1e-3 if not use_mapped_temperature else get_Temp_correction(temps_mK[-1])[0]*1e-3
            
            ## Get the file lists
            sum_file, dly_file, vna_file, nse_files, led_files = kipd.get_files(series, 
                                                                base_path=datapath,
                                                                sep_noise_pulse=True,
                                                                verbose=False)
            
            ## Parse the metadata
            voltages, p_params, charFs, charZs = kipd.parse_calibration_metadata(sum_file, 
                                                                blank_fraction=blank_fraction, 
                                                                verbose=False)
            
            ## Pull the VNA data
            VNA_f, VNA_z = kipd.read_vna(vna_file)
    
            ## Collect the VNA fit results: fr, Qr, Qi, Qc
            res_pars, res_errs = kipd.finefit(VNA_f/1e3, VNA_z, [4.242], restrict_fit_MHz=None, show_plots=False, verbose=False)
            
            ## Find the right LED file
            # led_file = led_files[np.argmin(np.abs(np.array(voltages)-this_V))]
            led_file = np.array(led_files)[[ (f"{this_V:.3f}V" in file) for file in led_files ]][0]
            print(led_file.split('/')[-1], temps_mK[-1], p_params["rf_power"])
            
            if not (p_params["rf_power"] == RF_power):
                j += 1
                continue
            
            ## Open the cleaned data and pull the data sampling rate
            clean_pulse_file   = led_file[:-3] + '_cleaned.h5'
            pulse_avg          = kipd.get_avgpulse_shape_from_file(clean_pulse_file)
            _, clean_data_info = kipd.get_cleaned_timestream(clean_pulse_file)
            
            sampling_rate = clean_data_info['sampling_rate']
        
            ## Define the time window for the pulse-full region, in microseconds
            time_window_range = fraction_to_keep * p_params['time_btw_pulse'] *1e6
            time_window = np.arange(0,time_window_range,1/sampling_rate*1e6)#[:-1]
            
            ## Convert the average pulses into the resonator basis
            frequency, dissipation, ideal, resonator = kipd.resonator_basis(pulse_avg,
                VNA_f=VNA_f,
                VNA_z=VNA_z,
                readout_f=charFs[0][0].real if len(np.shape(charFs))>1 else charFs[0].real,
                char_f=charFs[0].real if len(np.shape(charFs))>1 else charFs.real, 
                char_z=charZs[0] if len(np.shape(charFs))>1 else charZs)
            
            ## Convert the average pulses into the quasiparticle basis
            dnqp_k1, dnqp_k2 = kipd.quasiparticle_basis(frequency,dissipation,
                data_T= T_value_K,
                MB_results=MB_fit_vals,
                readout_f=charFs[0][0].real if len(np.shape(charFs))>1 else charFs[0].real)
            
            ## Save the time and pulse data
            t_vals = np.arange(len(pulse_avg))/sampling_rate*1e3
            p_vals = np.angle(pulse_avg)-np.mean(np.angle(pulse_avg)[0:100])
    
            ## Create a group for this temperature and save a bunch of metadata
            this_Tgrp = this_grp.create_group(str(temps_mK[-1])+"mK")
            this_Tgrp.attrs["T_sp(mK)"]  = temps_mK[-1]
            this_Tgrp.attrs["T_rf(mK)"]  = T_value_K * 1e3
            this_Tgrp.attrs["P_rf(dBm)"] = p_params["rf_power"]
            this_Tgrp.attrs["rate"] = sampling_rate

            this_Tgrp.attrs["fr(GHz)"] = res_pars["f0"]
            this_Tgrp.attrs["Qr"]      = res_pars["Qr"]
            this_Tgrp.attrs["Qc"]      = res_pars["Qc"]
            this_Tgrp.attrs["QcHat"]   = res_pars["QcHat"]
            this_Tgrp.attrs["phi"]     = res_pars["phi"]
            this_Tgrp.attrs["tau"]     = res_pars["tau"]
            this_Tgrp.attrs["zOff"]    = res_pars["zOff"]
    
            ## Save the time values
            this_Tgrp.create_dataset("time(ms)", data=t_vals)
    
            ## Save the pulse values 
            this_Tgrp.create_dataset("avg", data=p_vals)
            this_Tgrp.create_dataset("cplx", data=pulse_avg)
            this_Tgrp.create_dataset("freq", data=frequency   -np.mean(frequency[0:100]))
            this_Tgrp.create_dataset("diss", data=dissipation -np.mean(dissipation[0:100]))
            this_Tgrp.create_dataset("kappa1", data=dnqp_k1   -np.mean(dnqp_k1[0:100]))
            this_Tgrp.create_dataset("kappa2", data=dnqp_k2   -np.mean(dnqp_k2[0:100]))